<a href="https://colab.research.google.com/github/Phionanamugga/Thesis/blob/feature1/BERT_TFF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Importing libraries
import torch # Core deep learning framework
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments # Hugging Face Transformers for tokenization, pre-trained models, and training utilities
from datasets import load_dataset # Hugging Face Datasets for loading and managing datasets
 # Utilities for measuring runtime and system performance
import time
import psutil
# Visualization libraries for plotting training metrics and results
import matplotlib.pyplot as plt
import seaborn as sns
# Scientific computing and data manipulation
import numpy as np
import pandas as pd
# CodeCarbon to track energy consumption and estimate carbon emissions during training
from codecarbon import EmissionsTracker
# Standard Python utilities
import os
import warnings
import nbformat # nbformat for working with Jupyter Notebook files programmatically

In [4]:
# Suppressing warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Suppressing Hugging Face parallelism warnings

In [ ]:
# Setting random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [6]:
# Device configuration
#  "cuda" if a CUDA-enabled GPU is available
#  otherwise fallback to "cpu"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Show which device is being used (CPU or GPU)
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")# Display the installed PyTorch version

# Print the quantization engines supported on this system
# (used for model compression and faster inference on CPUs)
print(f"Supported quantization engines: {torch.backends.quantized.supported_engines}")

Using device: cpu
PyTorch version: 2.8.0+cu126
Supported quantization engines: ['qnnpack', 'none', 'onednn', 'x86', 'fbgemm']


In [7]:
# --- CodeCarbon Configuration ---
# This ensures energy consumption and CO₂ emissions can be tracked during training.
# The settings written:
# - measure_power_secs=0.1 : frequency (in seconds) of power usage measurements
# - cpu_power=35: assumed CPU power draw in watts (fallback for estimation)
if not os.path.exists(".codecarbon.config"):
    with open(".codecarbon.config", "w") as f:
        f.write("[codecarbon]\nmeasure_power_secs=0.1\ncpu_power=35\n")  # TDP for Intel i7

In [8]:
# Model Selection from the Hugging Face Model Hub
# - "BERT-base"   → standard BERT model (uncased, base version)
# - "DistilBERT"  → a lighter, faster version of BERT with fewer parameters
# This setup makes it easy to switch between different transformer architectures

models = {
    'BERT-base': 'bert-base-uncased',
    'DistilBERT': 'distilbert-base-uncased'
}

In [9]:
# Dataset Preparation
# Load the SST-2 sentiment classification task from the GLUE benchmark.

train_dataset = load_dataset("glue", "sst2", split="train[:10%]")  # Training set: only the first 10% is used here to speed up fine-tuning experiments.
eval_dataset = load_dataset("glue", "sst2", split="validation")  # Evaluation set: full validation split is loaded to properly measure model performance.

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]